# 05 · Scale & communication  (Phase **P5** — experiments E3, E8)

Two feasibility questions for a *cross-region* federation:

1. **Does it scale?** (E3) Vary the number of regional clients **K ∈ {5, 10, 20, 50}** and the
   per-round **participation fraction** (`fraction_fit ∈ {0.2, 0.5, 1.0}`). This argues the same
   architecture would extend to many regional nodes with only a subset reporting each round.
2. **What does communication cost, and can we cut it?** (E8) Apply **real uplink compression**
   — top-k sparsification (10%, 1%) and 8-bit quantization — and measure the **accuracy-per-MB**
   trade-off. The compression genuinely perturbs the update, so its accuracy cost is real.

Metric: **global-test accuracy** (comparable to P2/P3), plus **total communication (MB)**. Moderate
label skew (α = 0.5) throughout. **Resumable** per `(K, fraction_fit, compression, seed)`.

> Runs reuse each other (e.g. K=20@full serves both the scale and participation studies), so the
> unique run count is small (~9 at one seed). Set `SEEDS=[42,43,44]` later for CIs.


## 1 · Environment + Drive

In [ ]:
import os, sys, subprocess

REPO_URL = "https://github.com/sgogoigh/Satellite-Image-Classification-Fed-Learning.git"
REPO_DIR = "/content/Satellite-Image-Classification-Fed-Learning"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)
os.chdir(REPO_DIR)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

sys.path.insert(0, os.path.join(REPO_DIR, "src"))
import torch, fedsat
from fedsat.utils import get_device
print("fedsat", fedsat.__version__, "| torch", torch.__version__, "| device:", get_device())
if not torch.cuda.is_available():
    print("WARNING: no GPU. Runtime > Change runtime type > T4 GPU, then re-run this cell.")


In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = "/content/drive/MyDrive/fedsat"
except Exception as e:
    print("Drive not mounted (", e, ") -> falling back to local ./artifacts")
    DRIVE = os.path.join(REPO_DIR, "artifacts")
os.makedirs(DRIVE, exist_ok=True)
print("Artifacts dir:", DRIVE)


## 2 · Config + grids

In [ ]:
from fedsat.config import ExperimentConfig
from fedsat.utils import get_device

BASE = ExperimentConfig(
    experiment_name="p5_scale_comm",
    dataset="eurosat_rgb", hf_repo="blanchon/EuroSAT_RGB",
    num_clients=10, input_size=64,
    backbone="resnet18", pretrained=True, norm="bn",
    optimizer="sgd", lr=0.01, momentum=0.9, weight_decay=5e-4, batch_size=64,
    local_epochs=1, num_workers=2, max_epochs=40, early_stop_patience=7,
    data_cache_dir=f"{DRIVE}/hf_cache",
    partition_dir=f"{DRIVE}/partitions",
    results_dir=f"{DRIVE}/results",
    device=get_device(),
)

ALPHA = 0.5                       # moderate label skew throughout
NUM_ROUNDS = 15
LOCAL_EPOCHS = 1
SEEDS = [42]                      # scale/comm trends are clear at 1 seed; add 43,44 for CIs

K_GRID = [5, 10, 20, 50]          # E3: number of regional clients (fraction_fit = 1.0)
PART_K = 20                       # E3: partial-participation study is run at this K
FF_GRID = [0.2, 0.5, 1.0]         # E3: per-round participation fraction
COMP_K = 10                       # E8: compression study is run at this K (fraction_fit = 1.0)
COMPRESS_GRID = [(None, 0.0), ("topk", 0.1), ("topk", 0.01), ("8bit", 0.0)]

RESULTS_DIR = f"{DRIVE}/results/p5_scale_comm"
print("P5: scale (K) + participation (fraction_fit) + compression (uplink)")


## 3 · Load EuroSAT (once)

In [ ]:
from fedsat.data import load_eurosat, integrity_gate

hf_ds, class_names, labels = load_eurosat(BASE.hf_repo, cache_dir=BASE.data_cache_dir)
stats = integrity_gate(class_names, labels, expected_classes=BASE.num_classes,
                       expected_total=BASE.expected_total)
print("data ok:", stats["total"], "images |", stats["num_classes"], "classes")


## 4 · Resumable results store

In [ ]:
import os, json
os.makedirs(RESULTS_DIR, exist_ok=True)
STORE = os.path.join(RESULTS_DIR, "results.jsonl")

def clabel(compress, ratio):
    if compress is None: return "none"
    if compress == "topk": return f"topk@{ratio}"
    return compress

def load_rows():
    if not os.path.exists(STORE):
        return []
    return [json.loads(l) for l in open(STORE) if l.strip()]

def is_done(K, ff, label, seed):
    return any(r.get("K") == K and r.get("fraction_fit") == ff and r.get("compress") == label
               and r.get("seed") == seed for r in load_rows())

def append_row(row):
    with open(STORE, "a") as f:
        f.write(json.dumps(row) + "\n")

print("results store:", STORE, "| existing rows:", len(load_rows()))


## 5 · The runner + the (deduplicated) run list

`run_one` trains FedAvg for one `(K, fraction_fit, compression, seed)` and records global-test
accuracy + total communication. Runs shared across the three studies (e.g. `K=20, full, none`) are
only executed once.


In [ ]:
from fedsat.fl import run_fedavg
from fedsat.data import build_partition, save_partition, load_partition
from fedsat.utils import set_seed
import time

def run_one(K, ff, compress, ratio, seed):
    label = clabel(compress, ratio)
    if is_done(K, ff, label, seed):
        return
    cfg = BASE.replace(num_clients=K, alpha=ALPHA, seed=seed)
    try:
        part = load_partition(cfg)
    except FileNotFoundError:
        part = build_partition(cfg, labels, class_names, data_hash=stats["data_hash"])
        save_partition(cfg, part)
    set_seed(seed)
    print(f"K={K} ff={ff} compress={label} seed={seed} ...", flush=True)
    t = time.time()
    _, hist, s = run_fedavg(cfg, hf_ds, part, class_names, num_rounds=NUM_ROUNDS,
                            local_epochs=LOCAL_EPOCHS, fraction_fit=ff,
                            compress=compress, compress_ratio=ratio, verbose=False)
    append_row({"K": K, "fraction_fit": ff, "compress": label, "seed": seed,
                "test_acc": s["test_accuracy_at_best"], "total_comm_mb": s["total_comm_mb"],
                "best_round": s["best_round"], "epoch_equiv": s["epoch_equivalents"]})
    print(f"    test_acc={s['test_accuracy_at_best']:.4f}  comm={s['total_comm_mb']:.0f}MB  "
          f"({time.time()-t:.0f}s)", flush=True)

runs = []
for K in K_GRID:                 # E3 scale
    runs.append((K, 1.0, None, 0.0))
for ff in FF_GRID:               # E3 participation (at PART_K)
    runs.append((PART_K, ff, None, 0.0))
for c, r in COMPRESS_GRID:       # E8 compression (at COMP_K)
    runs.append((COMP_K, 1.0, c, r))

seen, uniq = set(), []
for K, ff, c, r in runs:
    key = (K, ff, clabel(c, r))
    if key in seen:
        continue
    seen.add(key); uniq.append((K, ff, c, r))
print(f"{len(uniq)} unique configs x {len(SEEDS)} seeds")


## 6 · Run the sweep (resumable)

In [ ]:
for (K, ff, c, r) in uniq:
    for seed in SEEDS:
        run_one(K, ff, c, r, seed)
print("\nP5 SWEEP COMPLETE. rows:", len(load_rows()))


## 7 · E3 — scale (accuracy + communication vs number of clients)

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt

df = pd.DataFrame(load_rows())

scale = (df[(df["fraction_fit"] == 1.0) & (df["compress"] == "none") & (df["K"].isin(K_GRID))]
         .groupby("K")[["test_acc", "total_comm_mb"]].mean().reindex(K_GRID))
print(scale)

fig, ax1 = plt.subplots(figsize=(8, 5))
ax1.plot(scale.index.astype(str), scale["test_acc"], "-o", color="#2980b9", label="global-test acc")
ax1.set_xlabel("number of clients K"); ax1.set_ylabel("global-test accuracy", color="#2980b9")
ax1.set_ylim(0, 1); ax1.grid(alpha=0.3)
ax2 = ax1.twinx()
ax2.plot(scale.index.astype(str), scale["total_comm_mb"], "-s", color="#d35400", label="total comm (MB)")
ax2.set_ylabel("total communication (MB)", color="#d35400")
plt.title(f"Scale: accuracy & communication vs K (alpha={ALPHA}, full participation)")
plt.tight_layout(); plt.show()


## 8 · E3 — partial participation (fixed K = PART_K)

In [ ]:
part = (df[(df["K"] == PART_K) & (df["compress"] == "none") & (df["fraction_fit"].isin(FF_GRID))]
        .groupby("fraction_fit")[["test_acc", "total_comm_mb"]].mean().reindex(FF_GRID))
print(part)

fig, ax1 = plt.subplots(figsize=(8, 5))
ax1.plot([str(f) for f in part.index], part["test_acc"], "-o", color="#2980b9", label="acc")
ax1.set_xlabel(f"fraction_fit (K={PART_K})"); ax1.set_ylabel("global-test accuracy", color="#2980b9")
ax1.set_ylim(0, 1); ax1.grid(alpha=0.3)
ax2 = ax1.twinx()
ax2.plot([str(f) for f in part.index], part["total_comm_mb"], "-s", color="#d35400")
ax2.set_ylabel("total communication (MB)", color="#d35400")
plt.title(f"Partial participation: accuracy & communication vs fraction_fit (K={PART_K})")
plt.tight_layout(); plt.show()


## 9 · E8 — communication compression (accuracy vs MB)

In [ ]:
comp = (df[(df["K"] == COMP_K) & (df["fraction_fit"] == 1.0)]
        .groupby("compress")[["test_acc", "total_comm_mb"]].mean())
order = [clabel(c, r) for c, r in COMPRESS_GRID]
comp = comp.reindex([o for o in order if o in comp.index])
print(comp)

base_mb = float(comp.loc["none", "total_comm_mb"]) if "none" in comp.index else np.nan
fig, ax = plt.subplots(figsize=(8, 5))
for name, row in comp.iterrows():
    ax.scatter(row["total_comm_mb"], row["test_acc"], s=90)
    save = f"  ({base_mb / row['total_comm_mb']:.0f}x less)" if (base_mb == base_mb and name != "none") else ""
    ax.annotate(name + save, (row["total_comm_mb"], row["test_acc"]),
                textcoords="offset points", xytext=(8, 4), fontsize=9)
ax.set_xscale("log"); ax.set_xlabel("total communication (MB, log scale)")
ax.set_ylabel("global-test accuracy"); ax.set_ylim(0, 1); ax.grid(alpha=0.3)
ax.set_title(f"Communication compression trade-off (K={COMP_K}, full participation)")
plt.tight_layout(); plt.show()

if "none" in comp.index:
    print("\naccuracy-per-MB (higher = more communication-efficient):")
    for name, row in comp.iterrows():
        print(f"  {name:<12} acc={row['test_acc']:.4f}  comm={row['total_comm_mb']:.1f}MB  "
              f"acc/MB={row['test_acc']/row['total_comm_mb']:.4f}")


## 10 · Save tables

In [ ]:
df.to_csv(os.path.join(RESULTS_DIR, "p5_summary.csv"), index=False)
print("saved ->", RESULTS_DIR)
print("\nScale (K):"); print(scale.to_string())
print("\nParticipation:"); print(part.to_string())
print("\nCompression:"); print(comp.to_string())


## Done — P5 complete

- **Scale (E3):** accuracy vs K shows the federation still trains as clients multiply (each holding
  less data); communication grows with K — the cost of scale, quantified.
- **Partial participation (E3):** lower `fraction_fit` cuts per-round communication substantially,
  with a measured accuracy trade-off — the lever that makes many-node federations practical.
- **Compression (E8):** top-k / 8-bit uplink compression shows the **accuracy-per-MB** curve — how
  much communication can be removed before accuracy suffers (the honest version of the old project's
  unsupported "communication efficiency" claim).

**Next — Phase P6 (`06_loco_generalization.ipynb`):** leave-one-client-out — train the federation on
K−1 regions and evaluate the global model on the unseen region's test set, quantifying cross-domain
generalization to a region never seen in training. (Optional: multispectral EuroSAT_MSI, and the
Track-B multi-dataset cross-domain study.)
